In [1]:
!python -m pip install --user --upgrade pip

In [2]:
# Ignore the warnings
import warnings
# warnings.filterwarnings('always')
warnings.filterwarnings('ignore')

# System related and data input controls
import os

# Data manipulation and visualization
import pandas as pd
pd.options.display.float_format = '{:,.2f}'.format
pd.options.display.max_rows = 20
pd.options.display.max_columns = 20
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import preprocessing

# Modeling algorithms
# General
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from scipy import stats

# Model selection
from sklearn.model_selection import train_test_split

# Evaluation metrics
# for classification
from sklearn import metrics
import geopandas as gpd
import math

In [3]:
# Classification Data (Using Direct Location)
location = 'D:/Analysis/BigData-PythonAnalysis-main/서울시_집계구_보행.csv'
ped_data = pd.read_csv(location,encoding="EUC-KR")
ped_data

,TOT_REG_CD,공공기관,보육시설,사회복지시,버스정류장수,버스승하차,지하철(1km이내),지하철거리점수,지하철승하차,구별총생활,구별총생활인구,상가수,어린이집개수,면적,병원수
0,1101053010001,6,4,0,3,196.56,4,24.00,0,277143,"8,940.10",7.00,6.00,18821,0
1,1101053010002,7,2,0,2,428.18,3,27.00,52315,1373417,"44,303.77",74.00,7.00,37082,0
2,1101053010003,10,6,0,12,497.20,4,26.00,34690,1245899,"40,190.29",37.00,10.00,218981,0
3,1101053010004,10,2,0,5,"3,152.08",4,29.00,52315,1515584,"48,889.81",113.00,10.00,44920,0
4,1101053010005,11,2,0,6,"2,136.02",3,29.00,52315,1038484,"33,499.48",122.00,11.00,70996,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19148,1125074030501,2,2,4,0,0.00,3,26.00,27891,231764,"7,476.26",13.00,2.00,2711,0
19149,1125074030701,2,1,2,0,0.00,3,28.00,27891,651864,"21,027.87",0.00,2.00,3522,0
19150,1125074030801,3,2,3,0,0.00,3,27.00,27891,303687,"9,796.35",9.00,3.00,5538,0
19151,1125074031101,3,2,3,0,0.00,3,26.00,27891,115108,"3,713.16",8.00,3.00,3535,0


In [8]:
ped_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19153 entries, 0 to 19152
Data columns (total 15 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   TOT_REG_CD  19153 non-null  int64  
 1   공공기관        19153 non-null  int64  
 2   보육시설        19153 non-null  int64  
 3   사회복지시       19153 non-null  int64  
 4   버스정류장수      19153 non-null  int64  
 5   버스승하차       19153 non-null  float64
 6   지하철(1km이내)  19153 non-null  int64  
 7   지하철거리점수     19143 non-null  float64
 8   지하철승하차      19153 non-null  int64  
 9   구별총생활       19153 non-null  int64  
 10  구별총생활인구     19153 non-null  float64
 11  상가수         19143 non-null  float64
 12  어린이집개수      18738 non-null  float64
 13  면적          19153 non-null  int64  
 14  병원수         19153 non-null  int64  
dtypes: float64(5), int64(10)
memory usage: 2.2 MB


In [4]:
ped_data.rename(columns={'사회복지시':'SOC_NUM','지하철(1km이내)':'SUB_NUM','지하철거리점수':'SUB_DIS_POINT'
 ,'버스정류장수':'BUS_NUM','버스승하차':'BUS_AVG','지하철승하차':'SUB_AVG'
 ,'구별총생활':'TOT_POP_GU','구별총생활인구':'TOT_POP_GU_AVG','상가수':'SHOP_NUM'
 , '어린이집개수':'CHILD_NUM','면적':'AREA','병원수':'HOSP_NUM'}
 ,inplace=True)

In [5]:
ped_data.isnull().sum()

TOT_REG_CD          0
공공기관                0
보육시설                0
SOC_NUM             0
BUS_NUM             0
BUS_AVG             0
SUB_NUM             0
SUB_DIS_POINT      10
SUB_AVG             0
TOT_POP_GU          0
TOT_POP_GU_AVG      0
SHOP_NUM           10
CHILD_NUM         415
AREA                0
HOSP_NUM            0
dtype: int64

In [5]:
ped_data['SUB_DIS_POINT'].fillna(0,inplace=True)
ped_data['SUB_DIS_POINT'].isnull().sum()
ped_data['SHOP_NUM'].fillna(0,inplace=True)
ped_data['SHOP_NUM'].isnull().sum()
ped_data['CHILD_NUM'].fillna(0,inplace=True)
ped_data['SHOP_NUM'].isnull().sum()

0

In [6]:
ped_data['FACILITY_NUM'] = (ped_data['공공기관']+ped_data['보육시설']+ped_data['CHILD_NUM']+ped_data['SOC_NUM'])

In [7]:
ped_data = ped_data.drop(['공공기관','보육시설','CHILD_NUM','SOC_NUM','TOT_REG_CD'],axis = 1)

In [9]:
display(ped_data.columns)

Index(['BUS_NUM', 'BUS_AVG', 'SUB_NUM', 'SUB_DIS_POINT', 'SUB_AVG',
       'TOT_POP_GU', 'TOT_POP_GU_AVG', 'SHOP_NUM', 'AREA', 'HOSP_NUM',
       'FACILITY_NUM'],
      dtype='object')

In [26]:
ped_data.head()

,BUS_NUM,BUS_AVG,SUB_NUM,SUB_DIS_POINT,SUB_AVG,TOT_POP_GU,TOT_POP_GU_AVG,SHOP_NUM,AREA,HOSP_NUM,FACILITY_NUM
0,3,196.56,4,24.00,0,277143,"8,940.10",7.00,18821,0,16.00
1,2,428.18,3,27.00,52315,1373417,"44,303.77",74.00,37082,0,16.00
2,12,497.20,4,26.00,34690,1245899,"40,190.29",37.00,218981,0,26.00
3,5,"3,152.08",4,29.00,52315,1515584,"48,889.81",113.00,44920,0,22.00
4,6,"2,136.02",3,29.00,52315,1038484,"33,499.48",122.00,70996,0,24.00


# 한달인구

In [10]:
Y_colname = ['TOT_POP_GU']
X_colname = ['BUS_NUM','SUB_NUM', 'SUB_DIS_POINT', 'SHOP_NUM', 'HOSP_NUM','FACILITY_NUM']
Y_colname, X_colname

(['TOT_POP_GU'],
 ['BUS_NUM',
  'SUB_NUM',
  'SUB_DIS_POINT',
  'SHOP_NUM',
  'HOSP_NUM',
  'FACILITY_NUM'])

In [34]:
X_train, X_test, Y_train, Y_test = train_test_split(ped_data[X_colname], ped_data[Y_colname],
                                                    test_size=0.2, random_state=123)
print(X_train.shape, Y_train.shape)
print(X_test.shape, Y_test.shape)

(15322, 6) (15322, 1)
(3831, 6) (3831, 1)


In [35]:
display(X_train, Y_train)

,BUS_NUM,SUB_NUM,SUB_DIS_POINT,SHOP_NUM,HOSP_NUM,FACILITY_NUM
5692,4,3,23.00,6.00,0,36.00
17561,1,5,27.00,10.00,0,24.00
10544,4,2,21.00,1.00,0,8.00
17368,3,4,25.00,56.00,0,6.00
5877,0,3,24.00,2.00,0,14.00
...,...,...,...,...,...,...
13435,2,2,26.00,1.00,0,7.00
7763,1,1,25.00,5.00,0,26.00
15377,0,3,27.00,1.00,0,10.00
17730,0,3,27.00,0.00,0,10.00


,TOT_POP_GU
5692,202303
17561,446774
10544,215643
17368,959127
5877,20636
...,...
13435,62674
7763,304265
15377,320613
17730,253567


In [36]:
X_train.describe(include='all').T

,count,mean,std,min,25%,50%,75%,max
BUS_NUM,"15,322.00",1.78,2.56,0.00,0.00,1.00,3.00,42.00
SUB_NUM,"15,322.00",2.41,1.45,0.00,1.00,2.00,3.00,12.00
SUB_DIS_POINT,"15,322.00",24.44,3.94,0.00,23.00,25.00,27.00,29.00
SHOP_NUM,"15,322.00",16.57,46.97,0.00,1.00,5.00,17.00,"1,795.00"
HOSP_NUM,"15,322.00",0.03,0.19,0.00,0.00,0.00,0.00,4.00
FACILITY_NUM,"15,322.00",15.86,7.86,0.00,10.00,15.00,21.00,77.00


In [38]:
sm.OLS(Y_train, X_train).fit().summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:             TOT_POP_GU   R-squared (uncentered):                   0.689
Model:                            OLS   Adj. R-squared (uncentered):              0.689
Method:                 Least Squares   F-statistic:                              5664.
Date:                Wed, 17 Nov 2021   Prob (F-statistic):                        0.00
Time:                        09:25:22   Log-Likelihood:                     -2.2141e+05
No. Observations:               15322   AIC:                                  4.428e+05
Df Residuals:                   15316   BIC:                                  4.429e+05
Df Model:                           6                                                  
Covariance Type:            nonrobust                                                  
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
BUS_NUM        6.628e+04   1652.842     40.098      0.000     6.3e+04    6.95e+04
SUB_NUM        3.823e+04   2977.763     12.840      0.000    3.24e+04    4.41e+04
SUB_DIS_POINT  1423.1246    455.418      3.125      0.002     530.452    2315.797
SHOP_NUM       8579.9787     89.708     95.644      0.000    8404.141    8755.816
HOSP_NUM       2.454e+05   2.02e+04     12.129      0.000    2.06e+05    2.85e+05
FACILITY_NUM   1986.8462    474.625      4.186      0.000    1056.526    2917.167
==============================================================================
Omnibus:                    22013.521   Durbin-Watson:                   2.006
Prob(Omnibus):                  0.000   Jarque-Bera (JB):         37093814.367
Skew:                           7.942   Prob(JB):                         0.00
Kurtosis:                     243.521   Cond. No.                         282.
==============================================================================

Notes:
[1] R² is computed without centering (uncentered) since the model does not contain a constant.
[2] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [40]:
scaler = preprocessing.MinMaxScaler()
scaler_fit = scaler.fit(X_train)
X_train_fes = pd.DataFrame(scaler_fit.transform(X_train), 
                           index=X_train.index, columns=X_train.columns)
X_test_fes = pd.DataFrame(scaler_fit.transform(X_test), 
                          index=X_test.index, columns=X_test.columns)

In [43]:
algo_lr=sm.OLS(Y_train, X_train_fes).fit()
algo_lr.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:             TOT_POP_GU   R-squared (uncentered):                   0.689
Model:                            OLS   Adj. R-squared (uncentered):              0.689
Method:                 Least Squares   F-statistic:                              5664.
Date:                Wed, 17 Nov 2021   Prob (F-statistic):                        0.00
Time:                        09:29:41   Log-Likelihood:                     -2.2141e+05
No. Observations:               15322   AIC:                                  4.428e+05
Df Residuals:                   15316   BIC:                                  4.429e+05
Df Model:                           6                                                  
Covariance Type:            nonrobust                                                  
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
BUS_NUM        2.784e+06   6.94e+04     40.098      0.000    2.65e+06    2.92e+06
SUB_NUM        4.588e+05   3.57e+04     12.840      0.000    3.89e+05    5.29e+05
SUB_DIS_POINT  4.127e+04   1.32e+04      3.125      0.002    1.54e+04    6.72e+04
SHOP_NUM        1.54e+07   1.61e+05     95.644      0.000    1.51e+07    1.57e+07
HOSP_NUM       9.817e+05   8.09e+04     12.129      0.000    8.23e+05    1.14e+06
FACILITY_NUM    1.53e+05   3.65e+04      4.186      0.000    8.14e+04    2.25e+05
==============================================================================
Omnibus:                    22013.521   Durbin-Watson:                   2.006
Prob(Omnibus):                  0.000   Jarque-Bera (JB):         37093814.367
Skew:                           7.942   Prob(JB):                         0.00
Kurtosis:                     243.521   Cond. No.                         40.3
==============================================================================

Notes:
[1] R² is computed without centering (uncentered) since the model does not contain a constant.
[2] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [44]:
Y_trpred = algo_lr.predict(X_train_fes).values
Y_tepred = algo_lr.predict(X_test_fes).values

In [45]:
MAE = abs(Y_test.values.flatten() - Y_tepred).mean()
MSE = ((Y_test.values.flatten() - Y_tepred)**2).mean()
MAPE = (abs(Y_test.values.flatten() - Y_tepred)/Y_test.values.flatten()*100).mean()
pd.DataFrame([MAE, MSE, MAPE], index=['MAE', 'MSE', 'MAPE'], columns=['Score']).T

,MAE,MSE,MAPE
Score,"207,956.25","185,439,110,138.50",92.71


# 일평균인구

In [19]:
Y_colname = ['TOT_POP_GU_AVG']
X_colname = ['BUS_NUM','SUB_NUM', 'SHOP_NUM', 'HOSP_NUM','FACILITY_NUM','HOSP_NUM']
Y_colname, X_colname

(['TOT_POP_GU_AVG'],
 ['BUS_NUM', 'SUB_NUM', 'SHOP_NUM', 'HOSP_NUM', 'FACILITY_NUM', 'HOSP_NUM'])

In [20]:
X_train, X_test, Y_train, Y_test = train_test_split(ped_data[X_colname], ped_data[Y_colname],
                                                    test_size=0.2, random_state=123)
print(X_train.shape, Y_train.shape)
print(X_test.shape, Y_test.shape)

(15322, 6) (15322, 1)
(3831, 6) (3831, 1)


In [21]:
display(X_train, Y_train)

,BUS_NUM,SUB_NUM,SHOP_NUM,HOSP_NUM,FACILITY_NUM,HOSP_NUM
5692,4,3,6.00,0,36.00,0
17561,1,5,10.00,0,24.00,0
10544,4,2,1.00,0,8.00,0
17368,3,4,56.00,0,6.00,0
5877,0,3,2.00,0,14.00,0
...,...,...,...,...,...,...
13435,2,2,1.00,0,7.00,0
7763,1,1,5.00,0,26.00,0
15377,0,3,1.00,0,10.00,0
17730,0,3,0.00,0,10.00,0


,TOT_POP_GU_AVG
5692,"6,525.90"
17561,"14,412.06"
10544,"6,956.23"
17368,"30,939.58"
5877,665.68
...,...
13435,"2,021.74"
7763,"9,815.00"
15377,"10,342.35"
17730,"8,179.58"


In [22]:
scaler = preprocessing.MinMaxScaler()
scaler_fit = scaler.fit(X_train)
X_train_fes = pd.DataFrame(scaler_fit.transform(X_train), 
                           index=X_train.index, columns=X_train.columns)
X_test_fes = pd.DataFrame(scaler_fit.transform(X_test), 
                          index=X_test.index, columns=X_test.columns)

In [23]:
algo_lr=sm.OLS(Y_train, X_train_fes).fit()
algo_lr.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:         TOT_POP_GU_AVG   R-squared (uncentered):                   0.689
Model:                            OLS   Adj. R-squared (uncentered):              0.689
Method:                 Least Squares   F-statistic:                              6791.
Date:                Tue, 23 Nov 2021   Prob (F-statistic):                        0.00
Time:                        09:24:59   Log-Likelihood:                     -1.6880e+05
No. Observations:               15322   AIC:                                  3.376e+05
Df Residuals:                   15317   BIC:                                  3.376e+05
Df Model:                           5                                                  
Covariance Type:            nonrobust                                                  
================================================================================
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
BUS_NUM       9.006e+04   2238.324     40.236      0.000    8.57e+04    9.44e+04
SUB_NUM       1.729e+04    831.816     20.792      0.000    1.57e+04    1.89e+04
SHOP_NUM      4.956e+05   5181.501     95.649      0.000    4.85e+05    5.06e+05
HOSP_NUM      1.581e+04   1305.790     12.104      0.000    1.32e+04    1.84e+04
FACILITY_NUM  7342.2875    892.666      8.225      0.000    5592.557    9092.018
HOSP_NUM      1.581e+04   1305.790     12.104      0.000    1.32e+04    1.84e+04
==============================================================================
Omnibus:                    21866.368   Durbin-Watson:                   2.006
Prob(Omnibus):                  0.000   Jarque-Bera (JB):         36206702.480
Skew:                           7.836   Prob(JB):                         0.00
Kurtosis:                     240.629   Cond. No.                     2.62e+16
==============================================================================

Notes:
[1] R² is computed without centering (uncentered) since the model does not contain a constant.
[2] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[3] The smallest eigenvalue is 2.2e-30. This might indicate that there are
strong multicollinearity problems or that the design matrix is singular.
"""

In [24]:
Y_trpred = algo_lr.predict(X_train_fes).values
Y_tepred = algo_lr.predict(X_test_fes).values

In [25]:
import math

In [26]:
MAE = abs(Y_test.values.flatten() - Y_tepred).mean()
MSE = math.sqrt(((Y_test.values.flatten() - Y_tepred)**2).mean())
MAPE = (abs(Y_test.values.flatten() - Y_tepred)/Y_test.values.flatten()*100).mean()
pd.DataFrame([MAE, MSE, MAPE], index=['MAE', 'MSE', 'MAPE'], columns=['Score']).T

,MAE,MSE,MAPE
Score,"6,753.40","13,901.29",91.86


# No test,train

In [9]:
Y_colname = ['TOT_POP_GU_AVG']
X_colname = ['BUS_NUM','SUB_NUM', 'SHOP_NUM', 'HOSP_NUM','FACILITY_NUM','HOSP_NUM']
Y_colname, X_colname

(['TOT_POP_GU_AVG'],
 ['BUS_NUM', 'SUB_NUM', 'SHOP_NUM', 'HOSP_NUM', 'FACILITY_NUM', 'HOSP_NUM'])

In [10]:
X_ped, Y_ped= ped_data[X_colname], ped_data[Y_colname]
                                   

In [11]:
display(X_ped, Y_ped)

,BUS_NUM,SUB_NUM,SHOP_NUM,HOSP_NUM,FACILITY_NUM,HOSP_NUM
0,3,4,7.00,0,16.00,0
1,2,3,74.00,0,16.00,0
2,12,4,37.00,0,26.00,0
3,5,4,113.00,0,22.00,0
4,6,3,122.00,0,24.00,0
...,...,...,...,...,...,...
19148,0,3,13.00,0,10.00,0
19149,0,3,0.00,0,7.00,0
19150,0,3,9.00,0,11.00,0
19151,0,3,8.00,0,11.00,0


,TOT_POP_GU_AVG
0,"8,940.10"
1,"44,303.77"
2,"40,190.29"
3,"48,889.81"
4,"33,499.48"
...,...
19148,"7,476.26"
19149,"21,027.87"
19150,"9,796.35"
19151,"3,713.16"


In [12]:
scaler = preprocessing.MinMaxScaler()
scaler_fit = scaler.fit(X_ped)
X_ped_fes = pd.DataFrame(scaler_fit.transform(X_ped), index=X_ped.index, columns=X_ped.columns)

In [14]:
algo_lr=sm.OLS(Y_ped, X_ped_fes).fit()
algo_lr.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:         TOT_POP_GU_AVG   R-squared (uncentered):                   0.688
Model:                            OLS   Adj. R-squared (uncentered):              0.688
Method:                 Least Squares   F-statistic:                              8464.
Date:                Tue, 23 Nov 2021   Prob (F-statistic):                        0.00
Time:                        09:18:36   Log-Likelihood:                     -2.1078e+05
No. Observations:               19153   AIC:                                  4.216e+05
Df Residuals:                   19148   BIC:                                  4.216e+05
Df Model:                           5                                                  
Covariance Type:            nonrobust                                                  
================================================================================
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
BUS_NUM       9.388e+04   1968.249     47.695      0.000       9e+04    9.77e+04
SUB_NUM       1.811e+04    737.489     24.561      0.000    1.67e+04    1.96e+04
SHOP_NUM      5.476e+05   5150.546    106.318      0.000    5.38e+05    5.58e+05
HOSP_NUM      1.599e+04   1145.990     13.949      0.000    1.37e+04    1.82e+04
FACILITY_NUM  8041.0745    977.483      8.226      0.000    6125.123    9957.026
HOSP_NUM      1.599e+04   1145.990     13.949      0.000    1.37e+04    1.82e+04
==============================================================================
Omnibus:                    27421.251   Durbin-Watson:                   1.844
Prob(Omnibus):                  0.000   Jarque-Bera (JB):         40520846.376
Skew:                           7.945   Prob(JB):                         0.00
Kurtosis:                     227.773   Cond. No.                     1.24e+15
==============================================================================

Notes:
[1] R² is computed without centering (uncentered) since the model does not contain a constant.
[2] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[3] The smallest eigenvalue is 1.03e-27. This might indicate that there are
strong multicollinearity problems or that the design matrix is singular.
"""

In [33]:
X_ped.head()

,BUS_NUM,SUB_NUM,SHOP_NUM,HOSP_NUM,FACILITY_NUM,HOSP_NUM
0,3,4,7.00,0,16.00,0
1,2,3,74.00,0,16.00,0
2,12,4,37.00,0,26.00,0
3,5,4,113.00,0,22.00,0
4,6,3,122.00,0,24.00,0


In [16]:
Y_pred = algo_lr.predict(X_ped_fes).values
Y_pred

array([15946.67860201, 29900.54778394, 44833.98459385, ...,
        7836.90591455,  7572.74954767,  5733.88267927])

In [18]:
MAE = abs(Y_ped.values.flatten() - Y_pred).mean()
MSE = math.sqrt(((Y_ped.values.flatten() - Y_pred)**2).mean())
MAPE = (abs(Y_ped.values.flatten() - Y_pred)/Y_ped.values.flatten()*100).mean()
pd.DataFrame([MAE, MSE, MAPE], index=['MAE', 'RMSE', 'MAPE'], columns=['Score']).T

,MAE,RMSE,MAPE
Score,"6,806.70","14,562.01",88.51


In [109]:
Y_pred = pd.DataFrame(Y_pred.flatten(), columns = ['prediction'])


In [110]:
Y_pred.to_csv('D:/Analysis/BigData-PythonAnalysis-main/OLS_pred.csv',encoding = 'utf8')

In [112]:
Y_pred.max()

prediction   587,365.24
dtype: float64